# Semana 5: agrupamiento de libros con Goodreads

Aplicamos K-Means para explorar perfiles de libros seg?n extensi?n, calificaci?n y popularidad. Los grupos son segmentos descriptivos, no categor?as reales ni una predicci?n. Se compara el efecto de escalar las variables y se elige `k` usando silhouette en una muestra reproducible.

## 1. Carga y diagn?stico de calidad

La auditor?a identifica faltantes y p?ginas extremas. Para la segmentaci?n se usa el percentil 99 de p?ginas positivas y se transforman los conteos de calificaciones/rese?as con `log1p`, porque suelen tener colas largas.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "goodreads.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "goodreads.py").exists():
    raise FileNotFoundError("No se encontr? src/goodreads.py. Abre el notebook desde el proyecto.")
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from goodreads import goodreads_quality_audit, prepare_goodreads_frame
from hf_loader import HFDatasetLoader

SEMILLA = 42

loader = HFDatasetLoader(dataset_name="pfaha/goodreads-books", split="train")
df_original = loader.load()
display(goodreads_quality_audit(df_original))
books = prepare_goodreads_frame(
    df_original, minimum_rating=3.5, maximum_rating=5.0, deduplicate_text=True
)
print(f"Libros elegibles: {len(books):,}")

## 2. Preparar variables comparables

Los valores no v?lidos de conteos se tratan como faltantes y se imputan con la mediana. `StandardScaler` evita que la escala de p?ginas o popularidad domine las distancias. El conjunto de muestra se usa tanto para evaluar `k` como para describir los segmentos.

In [ ]:
feature_columns = ["paginas", "calificacion", "cantidad_calificaciones", "cantidad_resenas"]
missing_features = set(feature_columns).difference(books.columns)
if missing_features:
    raise KeyError(f"Faltan variables para segmentar: {sorted(missing_features)}")

paginas_positivas = books.loc[books["paginas"].gt(0), "paginas"]
limite_paginas = float(paginas_positivas.quantile(0.99))
segment_data = books[
    books["paginas"].isna() | books["paginas"].between(1, limite_paginas)
].copy()
X = segment_data[feature_columns].apply(pd.to_numeric, errors="coerce")
for column in ["cantidad_calificaciones", "cantidad_resenas"]:
    X.loc[X[column] < 0, column] = np.nan
    X[column] = np.log1p(X[column])
segment_data = segment_data.loc[X.notna().any(axis=1)].copy()
X = X.loc[segment_data.index]

sample_size = min(20_000, len(X))
segment_data = segment_data.sample(n=sample_size, random_state=SEMILLA)
X = X.loc[segment_data.index]
print(f"Muestra: {len(X):,}; l?mite de p?ginas P99: {limite_paginas:.0f}")

## 3. Selecci?n de `k` y efecto de la escala

El silhouette se calcula en una submuestra de 2.000 libros para mantener acotado el costo cuadr?tico. Se reportan los valores para `k=2..6` tanto con estandarizaci?n como sin ella; la selecci?n principal usa datos escalados.

In [ ]:
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

scores = []
models = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(
        X_scaled, labels, sample_size=min(2_000, len(X_scaled)), random_state=SEMILLA
    )
    scores.append({"k": k, "silhouette_escalado": score, "inercia": model.inertia_})
    models[k] = model
scores = pd.DataFrame(scores).sort_values("k").reset_index(drop=True)
display(scores.round(4))
k_elegido = int(scores.loc[scores["silhouette_escalado"].idxmax(), "k"])

labels_raw = KMeans(n_clusters=k_elegido, n_init=10, random_state=SEMILLA).fit_predict(X_imputed)
silhouette_raw = silhouette_score(
    X_imputed, labels_raw, sample_size=min(2_000, len(X_imputed)), random_state=SEMILLA
)
print(f"k elegido: {k_elegido}; silhouette escalado: {scores.loc[scores.k == k_elegido, 'silhouette_escalado'].iloc[0]:.4f}")
print(f"Silhouette sin escalar para el mismo k: {silhouette_raw:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(scores["k"], scores["silhouette_escalado"], marker="o", label="Silhouette escalado")
ax.set(xlabel="N?mero de grupos (k)", ylabel="Silhouette", title="Selecci?n de k")
ax.set_xticks(scores["k"])
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Interpretar y guardar segmentos

Se describen tama?o, calificaci?n media, p?ginas y conteos medios por grupo. Como los conteos son asim?tricos, la tabla tambi?n incluye medianas para evitar que unos pocos libros muy populares dominen la interpretaci?n.

In [ ]:
from joblib import dump

pipeline = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler()),
    ("kmeans", KMeans(n_clusters=k_elegido, n_init=10, random_state=SEMILLA)),
])
segment_data["segmento"] = pipeline.fit_predict(X)
profile = segment_data.groupby("segmento").agg(
    libros=("titulo", "size"),
    calificacion_media=("calificacion", "mean"),
    paginas_mediana=("paginas", "median"),
    calificaciones_mediana=("cantidad_calificaciones", "median"),
    resenas_mediana=("cantidad_resenas", "median"),
).round(2)
display(profile)

report_path = ROOT / "reportes" / "resumen_segmentacion_libros.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
model_path = ROOT / "modelos" / "libros" / f"{datetime.now():%Y%m%d_%H%M%S}_kmeans_goodreads.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
dump(pipeline, model_path)
report_path.write_text(
    "SEGMENTACI?N DE LIBROS GOODREADS\n"
    "================================\n"
    f"Filas originales: {len(df_original):,}; libros elegibles: {len(books):,}\n"
    f"Muestra: {len(segment_data):,}; semilla: {SEMILLA}; l?mite p?ginas P99: {limite_paginas:.2f}\n"
    f"Variables: {feature_columns}; popularidad transformada con log1p; imputaci?n mediana y escalado est?ndar.\n"
    f"Silhouette sin escalar (k={k_elegido}): {silhouette_raw:.4f}\n"
    f"Resultados de k:\n{scores.round(4).to_string(index=False)}\n\n"
    f"Perfil de grupos:\n{profile.to_string()}\n"
    "Nota: estos grupos describen la muestra; no son clases naturales garantizadas.\n",
    encoding="utf-8",
)
print(f"Reporte: {report_path}")
print(f"Modelo guardado (ignorado por Git): {model_path}")